# 1. Normalización de la tabla de Manufactura

Vamos a aplicar las primeras 4 formas normales en los siguientes datos:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\Proyectos Big data\1. Manufactura\archive\Intelligent_Production_IIoT.csv")
df.head()

,record_id,machine_id,production_line,batch_id,shift,product_type,ambient_temperature,humidity,air_pressure,ambient_vibration,...,downtime_minutes,maintenance_frequency,production_cost_per_unit,resource_efficiency,production_efficiency,energy_saving_pct,downtime_reduction_pct,cost_reduction_pct,benefit_score,operation_mode
0,1,M07,Line_A,B0343,Morning,Product_C,24.657935,69.272037,102.335679,0.366190,...,25.081059,2.172029,8.028928,99.0,99.0,9.364688,49.353148,26.199425,68.260235,High_Throughput
1,2,M20,Line_E,B0593,Afternoon,Product_D,15.953337,59.927354,100.055317,0.472410,...,20.400191,2.576074,8.537013,99.0,99.0,3.129223,56.434784,30.016525,68.491891,High_Throughput
2,3,M15,Line_E,B0205,Afternoon,Product_C,21.761215,44.487079,102.794636,0.478138,...,11.608597,2.357031,7.372440,99.0,99.0,6.433856,61.539859,33.412663,68.837230,Balanced_Production
3,4,M11,Line_B,B0993,Morning,Product_A,21.973528,59.022870,98.258859,0.417760,...,25.542778,2.840125,8.622625,99.0,99.0,3.284783,43.948708,34.699694,67.074754,Balanced_Production
4,5,M08,Line_E,B0961,Night,Product_C,20.595578,54.665654,99.217605,0.283555,...,19.759181,3.410520,7.521289,99.0,99.0,12.958237,52.025084,31.553079,69.084109,Precision_Quality


### Análisis exploratorio de datos

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   record_id                 15000 non-null  int64  
 1   machine_id                15000 non-null  str    
 2   production_line           15000 non-null  str    
 3   batch_id                  15000 non-null  str    
 4   shift                     15000 non-null  str    
 5   product_type              15000 non-null  str    
 6   ambient_temperature       14611 non-null  float64
 7   humidity                  14599 non-null  float64
 8   air_pressure              14620 non-null  float64
 9   ambient_vibration         15000 non-null  float64
 10  motor_temperature         14639 non-null  float64
 11  spindle_speed             14642 non-null  float64
 12  motor_current             14616 non-null  float64
 13  torque                    14613 non-null  float64
 14  vibration        

## 1 Primera Forma Normal (1NF)

Una tabla está en 1NF si cumple con los siguientes criterios:

- Cada celda contiene un único valor indivisible (atomicidad).

- No hay grupos repetitivos ni valores múltiples en una celda.

- Existe una clave primaria que identifica de manera única cada fila.

Ejemplo: Si una tabla almacena colores disponibles en una sola celda como "Rojo, Azul, Verde", debe dividirse en filas individuales para cada color.

In [4]:
# 1. Verificar si 'record_id' es una clave primaria válida (sin nulos y sin duplicados)
nulos = df['record_id'].isnull().sum()
duplicados = df['record_id'].duplicated().sum()

print(f"Valores nulos en record_id: {nulos}")
print(f"Registros duplicados en record_id: {duplicados}")

# 2. Verificar atomicidad (ejemplo: buscar si alguna celda de texto tiene comas u objetos extraños)
# Si da 0, significa que no hay listas aparentes guardadas como texto plano.
celdas_con_comas = df.select_dtypes(include=['object']).apply(lambda x: x.str.contains(',').sum()).sum()
print(f"Posibles valores múltiples detectados por comas: {celdas_con_comas}")

Valores nulos en record_id: 0
Registros duplicados en record_id: 0
Posibles valores múltiples detectados por comas: 0


C:\Users\ASUS\AppData\Local\Temp\ipykernel_15132\1209066657.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  celdas_con_comas = df.select_dtypes(include=['object']).apply(lambda x: x.str.contains(',').sum()).sum()


## 2 Segunda Forma Normal (2NF)

Una tabla está en 2NF si:

Cumple con 1NF.

Todos los atributos no clave dependen completamente de la clave primaria, eliminando dependencias parciales.

Ejemplo: Si una tabla tiene una clave primaria compuesta y un atributo depende solo de una parte de esa clave, debe dividirse en tablas separadas para eliminar esta dependencia.

## 2.1  Separación de las Subtablas de normalización

In [5]:
# 1. Crear la subtabla / catálogo de Máquinas
df_maquinas = df[['machine_id', 'production_line', 'maintenance_frequency']].drop_duplicates().reset_index(drop=True)

# 2. Crear la subtabla / catálogo de Productos
df_productos = df[['product_type', 'production_cost_per_unit']].drop_duplicates().reset_index(drop=True)

# 3. Dejar la tabla principal (Hechos / Producción) solo con las métricas y las llaves foráneas
df_produccion_principal = df[[
    'record_id', 'machine_id', 'product_type', 'batch_id', 'shift',
    'ambient_temperature', 'humidity', 'spindle_speed', 'power_consumption',
    'defect_rate', 'downtime_minutes', 'operation_mode'
]]

# Guardar el resultado normalizado en un archivo de Excel con múltiples hojas
with pd.ExcelWriter('manufactura_normalizada.xlsx') as writer:
    df_produccion_principal.to_excel(writer, sheet_name='Metricas_Produccion', index=False)
    df_maquinas.to_excel(writer, sheet_name='Catalogo_Maquinas', index=False)
    df_productos.to_excel(writer, sheet_name='Catalogo_Productos', index=False)

## 3 Tercera Forma Normal (3NF)

Una tabla está en 3NF si:

- Cumple con 2NF.

- No existen dependencias transitivas, es decir, un atributo no clave no depende de otro atributo no clave.

Ejemplo: Si una tabla incluye el nombre de un departamento que depende de un identificador de departamento, esta información debe moverse a una tabla separada.